# Logistische Regression (Optimierung der negativen LogLikelihood-Funktion)

In [229]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [230]:
from datasets import load_dataset
dataset = load_dataset("sms_spam")
dataset

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

In [231]:
dataset["train"]["sms"][2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n"

In [232]:
dataset["train"]["label"][2]

1

In [233]:
len(dataset["train"]["sms"]) # 5574 SMS 

5574

$x \in \mathbb{R}^d$

In [234]:
dataset["train"]["sms"][2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n"

In [235]:
dataset["train"]["sms"][9]

'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030\n'

1. ASCII
    - einfach, Alphabet ist klein (256 Zeichen)
    - Nachteil: sehr lange Sequenzen 
2. Tokenisierung
    - " Up" -> 10
    - "date " -> 22 
    - Update = [10,22]
    - Nachteil: immer noch eine Sequenz 
3. Vektorisierung
    - ganze Wörter als Einträge in den Vektor

In [236]:
doc1 = "large string"
doc2 = "second large string"
doc3 = "third very very large string"

| x | large | string | second | third | very | Typ |
|---|-------|--------|--------|-------|------|-----|
| doc1 | 1  |   1    |    0   |   0   |  0   |  binary |
| doc3 | 1   |  1    |    0   |    1  |  2   |  count |
| doc3 | 1   |  1    |    0   |    1  |  1   |  binary |

In [237]:
from sklearn.model_selection import train_test_split

In [238]:
X = dataset["train"].train_test_split(test_size=0.2) # 80/20 split 

In [250]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=1000,binary=True) # 1000 häufigste Wörter
X_train = vectorizer.fit_transform(x["sms"] for x in X["train"]).toarray()
X_test = vectorizer.transform(x["sms"] for x in X["test"]).toarray()
y_train = [x["label"] for x in X["train"]]
y_test = [x["label"] for x in X["test"]]

In [ ]:
# Data Leakage 
# dataset["train"] # ursprünglicher Datensatz
# vectorizer = CountVectorizer(max_features=1000,binary=True)
# X_falsch = vectorizer.fit_transform(x["sms"] for x in dataset["train"]).toarray()
# X_train_f, X_test_f = train_test_split(X_falsch, test_size=0.2)

In [240]:
import numpy as np

X_train_np = X_train.astype(np.float32)
X_test_np = X_test.astype(np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

# z = b*1 + w1*x1 + ... + wd*xd
# Bias 
X_train_b = np.hstack([np.ones((X_train_np.shape[0],1)), X_train_np])
X_test_b = np.hstack([np.ones((X_test_np.shape[0],1)), X_test_np])
X_train_b

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(4459, 1001))

In [241]:
len(X_train_np)

4459

In [242]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

alpha = 0.1 # Lernrate 
n_epochs = 200 # 1 Epoche = 1 Mal hat ML Algorithmus den gesamten Datensatz "gesehen"
batch_size = 64 

n = len(X_train_b) 
w = np.random.randn(X_train_b.shape[1],1) # irgendwo zufällig 

w_start = w 
for epoch in range(n_epochs): # Training-Loop 
    shuffle_idx = np.random.permutation(n) # Mischen 
    X_b_shuffle = X_train_b[shuffle_idx]
    y_shuffle = y_train_np[shuffle_idx]
    for i in range(0, n - batch_size, batch_size): # von Null bis N in batch_size Schritten 
        xi = X_b_shuffle[i:i+batch_size] # Auswahl vom Datenpunkten 
        yi = y_shuffle[i:i+batch_size]
        # z = b*1 + w1*x1 + ... + wd*xd 
        logits = xi @ w # lineares Modell, logit = Rohwert vor der Aktivierung  
        probs = sigmoid(logits)
        
        gradients = xi.T @ (probs.T - yi).T
        w = w - alpha * gradients

In [243]:
test_probs = sigmoid(X_test_b @ w)
test_pred = (test_probs > 0.5).astype(np.int32)
acc = (test_pred.T == y_test_np).mean() # Genauigkeit
print(f"Genauigkeit: {acc*100:.2f}%")

Genauigkeit: 98.39%


In [244]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_pred, y_test_np)
cm

array([[967,  14],
       [  4, 130]])

In [245]:
949 + 28 # legitime Email


977

In [246]:
(test_pred.T == y_test_np).mean()

np.float64(0.9838565022421525)

In [247]:
def predict_spam(msg): 
    x = vectorizer.transform([msg]).toarray().astype(np.float32)
    x_b = np.hstack([np.ones((x.shape[0], 1), dtype=np.float32), x])
    prob = sigmoid(x_b @ w)[0]
    return "SPAM" if prob > 0.5 else "NO SPAM" 


print(predict_spam('Congratulations, you have won a prize!'))
print(predict_spam('Are we meeting at 7pm for dinner?'))

SPAM
NO SPAM
